# 📈 Spotify Streaming – Jahres-Trends

Monatliche Hörzeit pro Jahr, saisonale Muster und Year-over-Year-Vergleiche.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np
from config import *
from data_loader import load_data, get_music, ensure_results_dir

plt.style.use(PLOT_STYLE)
plt.rcParams['figure.dpi'] = PLOT_DPI
plt.rcParams['savefig.dpi'] = PLOT_DPI
plt.rcParams['savefig.bbox'] = 'tight'

In [ ]:
df = load_data()
music = get_music(df)

## Monatliche Hörzeit – Alle Jahre überlagert

In [ ]:
monthly = music.groupby(['year', 'month'])['hours_played'].sum().reset_index()
years = sorted(monthly['year'].unique())
month_names = ['Jan', 'Feb', 'Mär', 'Apr', 'Mai', 'Jun', 'Jul', 'Aug', 'Sep', 'Okt', 'Nov', 'Dez']

fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_WIDE)
for i, year in enumerate(years):
    year_data = monthly[monthly['year'] == year]
    ax.plot(year_data['month'], year_data['hours_played'],
            marker='o', label=str(year), linewidth=2,
            color=COLOR_PALETTE[i % len(COLOR_PALETTE)])

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)
ax.set_xlabel('Monat')
ax.set_ylabel('Stunden')
ax.set_title('🗓️ Monatliche Hörzeit pro Jahr', fontsize=16, fontweight='bold')
ax.legend(title='Jahr', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

out = ensure_results_dir('overview')
fig.savefig(out / 'monthly_hours_overlay.png')
plt.show()

## Monatliche Hörzeit – Pro Jahr einzeln (gespeichert in results/<year>/)

In [ ]:
for year in years:
    year_data = monthly[monthly['year'] == year]
    if len(year_data) < 2:
        continue

    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_WIDE)
    ax.fill_between(year_data['month'], year_data['hours_played'], alpha=0.3, color=COLOR_PRIMARY)
    ax.plot(year_data['month'], year_data['hours_played'],
            marker='o', linewidth=2.5, color=COLOR_PRIMARY)
    for _, row in year_data.iterrows():
        ax.annotate(f"{row['hours_played']:.0f}h",
                    (row['month'], row['hours_played']),
                    textcoords='offset points', xytext=(0, 10),
                    ha='center', fontsize=9)

    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(month_names)
    ax.set_xlabel('Monat')
    ax.set_ylabel('Stunden')
    ax.set_title(f'📅 Monatliche Hörzeit {year}', fontsize=16, fontweight='bold')
    ax.set_ylim(0, year_data['hours_played'].max() * 1.25)

    year_out = ensure_results_dir(year)
    fig.savefig(year_out / 'monthly_hours.png')
    plt.show()
    print(f"Gespeichert: {year_out / 'monthly_hours.png'}")

## Tägliche Hörzeit – Rolling Average

In [ ]:
daily = music.groupby('date')['minutes_played'].sum().reset_index()
daily['date'] = pd.to_datetime(daily['date'])
daily = daily.set_index('date').sort_index()
daily['rolling_7d'] = daily['minutes_played'].rolling(7, min_periods=1).mean()
daily['rolling_30d'] = daily['minutes_played'].rolling(30, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(daily.index, daily['rolling_7d'], linewidth=0.8, alpha=0.5, color=COLOR_PRIMARY, label='7-Tage Ø')
ax.plot(daily.index, daily['rolling_30d'], linewidth=2, color=COLOR_ACCENT, label='30-Tage Ø')
ax.fill_between(daily.index, daily['rolling_30d'], alpha=0.15, color=COLOR_PRIMARY)
ax.set_xlabel('Datum')
ax.set_ylabel('Minuten pro Tag')
ax.set_title('📊 Tägliche Hörzeit (gleitender Durchschnitt)', fontsize=16, fontweight='bold')
ax.legend()
plt.tight_layout()
fig.savefig(out / 'daily_rolling_average.png')
plt.show()

## Durchschnittliche Minuten pro Tag – Jahresvergleich

In [ ]:
daily_by_year = music.groupby(['year', 'date'])['minutes_played'].sum().reset_index()
avg_daily = daily_by_year.groupby('year')['minutes_played'].mean().reset_index()

fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_WIDE)
bars = ax.bar(avg_daily['year'].astype(str), avg_daily['minutes_played'], color=COLOR_ACCENT)
for bar, v in zip(bars, avg_daily['minutes_played']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{v:.0f} min', ha='center', fontweight='bold', fontsize=10)
ax.set_xlabel('Jahr')
ax.set_ylabel('Ø Minuten / Tag')
ax.set_title('⏱️ Durchschnittliche Hörzeit pro Tag', fontsize=16, fontweight='bold')
ax.set_ylim(0, avg_daily['minutes_played'].max() * 1.2)
fig.savefig(out / 'avg_daily_minutes_by_year.png')
plt.show()